# 04. 데이터 파이프라인

## 학습 목표
- LLM 학습에 사용되는 데이터 소스와 특성 이해
- 데이터 품질 관리: 중복 제거 (MinHash), 필터링 개념 파악
- HuggingFace Datasets로 데이터 로드, 전처리, 스트리밍 실습
- 텍스트 -> 토큰 IDs -> 고정 길이 청크 파이프라인 구축
- DataLoader 구성: 셔플, 배치, 패딩 전략

## 참고 자료
- [HuggingFace Datasets Documentation](https://huggingface.co/docs/datasets)
- [The Pile: An 800GB Dataset of Diverse Text](https://arxiv.org/abs/2101.00027)

---

In [ ]:
# Google Colab에서 필요한 패키지 설치
# !pip install datasets transformers torch matplotlib -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import hashlib
from collections import Counter

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False

## 1. LLM 학습 데이터 소스

LLM은 인터넷의 방대한 텍스트로 학습된다.

| 데이터 소스 | 크기 | 특성 | 사용 모델 |
|------------|------|------|----------|
| Common Crawl | ~PB | 웹 크롤링, 노이즈 많음 | GPT-3, LLaMA |
| Wikipedia | ~20GB | 고품질, 다국어 | 거의 모든 LLM |
| Books | ~100GB | 긴 문맥, 문학적 표현 | GPT-3, PaLM |
| Code (GitHub) | ~1TB | 프로그래밍 언어 | Codex, StarCoder |
| ArXiv | ~50GB | 과학 논문 | Galactica |
| StackExchange | ~50GB | Q&A 형식 | LLaMA |

### 데이터 혼합 비율 (예시: LLaMA)

LLaMA는 데이터 소스별로 다른 샘플링 비율을 적용했다.

In [ ]:
# LLaMA 학습 데이터 혼합 비율 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LLaMA 데이터 구성
llama_data = {
    'CommonCrawl': 67.0,
    'C4': 15.0,
    'GitHub': 4.5,
    'Wikipedia': 4.5,
    'Books': 4.5,
    'ArXiv': 2.5,
    'StackExchange': 2.0,
}

# 왼쪽: 파이 차트
ax = axes[0]
colors = plt.cm.Set3(np.linspace(0, 1, len(llama_data)))
wedges, texts, autotexts = ax.pie(
    llama_data.values(), labels=llama_data.keys(),
    autopct='%1.1f%%', colors=colors, startangle=90,
    textprops={'fontsize': 9}
)
ax.set_title('LLaMA Training Data Mix')

# 오른쪽: 샘플링 비율 (epochs)
ax = axes[1]
sampling = {
    'CommonCrawl': 1.10,
    'C4': 1.06,
    'GitHub': 0.64,
    'Wikipedia': 2.45,
    'Books': 2.23,
    'ArXiv': 1.06,
    'StackExchange': 1.03,
}

bars = ax.bar(sampling.keys(), sampling.values(), color=colors, edgecolor='black')
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='1 epoch')
ax.set_ylabel('Sampling Ratio (epochs)')
ax.set_title('LLaMA: Data Source Sampling Ratio')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=30, ha='right')

for bar, val in zip(bars, sampling.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,
            f'{val:.2f}x', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("핵심: 고품질 데이터(Wikipedia, Books)는 여러 번 반복 학습 (>1 epoch)")
print("     노이즈가 많은 데이터(GitHub)는 적게 사용 (<1 epoch)")

---
## 2. 데이터 품질 관리

### 2.1 중복 제거: MinHash

웹에서 수집한 데이터는 중복이 많다. 중복 데이터는:
- 모델이 특정 텍스트를 **외우게** 됨
- 평가 데이터에 학습 데이터가 섯여 들어가는 **데이터 오염**

**MinHash**: 문서의 유사도를 빠르게 추정하는 알고리즘

1. 문서를 n-gram 집합으로 변환
2. 여러 해시 함수로 각 n-gram을 해싱
3. 각 해시 함수의 **최소값**을 시그니처로 사용
4. 시그니처가 비슷한 문서 = 유사한 문서

**Jaccard 유사도**: 두 집합의 유사도

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

In [ ]:
# MinHash 간단 구현

def get_ngrams(text, n=3):
    """텍스트에서 n-gram 집합을 추출"""
    words = text.lower().split()
    return set([' '.join(words[i:i+n]) for i in range(len(words) - n + 1)])


def jaccard_similarity(set_a, set_b):
    """정확한 Jaccard 유사도 계산"""
    intersection = len(set_a & set_b)
    union = len(set_a | set_b)
    return intersection / union if union > 0 else 0


def minhash_signature(ngrams, num_hashes=100):
    """여러 해시 함수로 MinHash 시그니처 생성"""
    signature = []
    for i in range(num_hashes):
        min_hash = float('inf')
        for ngram in ngrams:
            # 해시 함수: hash(ngram + salt)
            h = int(hashlib.md5(f"{ngram}_{i}".encode()).hexdigest(), 16)
            min_hash = min(min_hash, h)
        signature.append(min_hash)
    return signature


def minhash_similarity(sig_a, sig_b):
    """두 MinHash 시그니처의 유사도 추정"""
    matches = sum(a == b for a, b in zip(sig_a, sig_b))
    return matches / len(sig_a)


# 테스트 문서들
doc1 = "오늘 날씨가 정말 좋습니다 하늘이 맑고 바람이 상쾌합니다"
doc2 = "오늘 날씨가 정말 좋아요 하늘이 맑고 바람이 상쾌해요"  # 거의 같은 문서
doc3 = "인공지능 기술이 빠르게 발전하고 있습니다"  # 다른 문서

# n-gram 추출
ng1 = get_ngrams(doc1, n=2)
ng2 = get_ngrams(doc2, n=2)
ng3 = get_ngrams(doc3, n=2)

print("\uc815\ud655\ud55c Jaccard \uc720\uc0ac\ub3c4:")
print(f"  doc1 vs doc2: {jaccard_similarity(ng1, ng2):.3f} (\uac70\uc758 \ub3d9\uc77c)")
print(f"  doc1 vs doc3: {jaccard_similarity(ng1, ng3):.3f} (\uc644\uc804\ud788 \ub2e4\ub984)")
print(f"  doc2 vs doc3: {jaccard_similarity(ng2, ng3):.3f} (\uc644\uc804\ud788 \ub2e4\ub984)")

# MinHash 시그니처
sig1 = minhash_signature(ng1, num_hashes=200)
sig2 = minhash_signature(ng2, num_hashes=200)
sig3 = minhash_signature(ng3, num_hashes=200)

print(f"\nMinHash \ucd94\uc815 \uc720\uc0ac\ub3c4 (200 hashes):")
print(f"  doc1 vs doc2: {minhash_similarity(sig1, sig2):.3f}")
print(f"  doc1 vs doc3: {minhash_similarity(sig1, sig3):.3f}")
print(f"  doc2 vs doc3: {minhash_similarity(sig2, sig3):.3f}")

print(f"\n-> MinHash\ub294 \uc815\ud655\ud55c Jaccard\ub97c \ucd94\uc815. \ud574\uc2dc \uc218\uac00 \ub9ce\uc744\uc218\ub85d \uc815\ud655.")
print(f"   \uc7a5\uc810: O(n) -> O(k)\ub85c \ube44\uad50 \uac00\ub2a5 (k = \ud574\uc2dc \uc218)")

In [ ]:
# MinHash 정확도 vs 해시 수 시각화
true_sim = jaccard_similarity(ng1, ng2)

hash_counts = [10, 20, 50, 100, 200, 500]
estimates = []
errors = []

for num_h in hash_counts:
    sims = []
    for trial in range(50):  # 50회 반복
        s1 = minhash_signature(ng1, num_hashes=num_h)
        s2 = minhash_signature(ng2, num_hashes=num_h)
        sims.append(minhash_similarity(s1, s2))
    estimates.append(np.mean(sims))
    errors.append(np.std(sims))

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(hash_counts, estimates, yerr=errors, fmt='bo-', capsize=5, linewidth=2, label='MinHash Estimate')
ax.axhline(y=true_sim, color='red', linestyle='--', linewidth=2, label=f'True Jaccard = {true_sim:.3f}')
ax.set_xlabel('Number of Hash Functions')
ax.set_ylabel('Estimated Jaccard Similarity')
ax.set_title('MinHash Accuracy vs Number of Hashes')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xscale('log')

plt.tight_layout()
plt.show()

print("-> 해시 수가 늘어날수록 추정치가 정확해지고 분산이 줄어든다")
print("   실제 LLM 학습에서는 128~256개 해시가 일반적")

### 2.2 데이터 품질 필터링

Common Crawl 등에서 수집한 데이터는 다양한 품질 문제가 있다:

| 필터 | 설명 | 방법 |
|------|------|------|
| 언어 감지 | 원하는 언어만 선택 | fastText langid |
| 길이 필터 | 너무 짧거나 긴 문서 제거 | 단어 수 기준 |
| 품질 필터 | 노이즈, 스팸 등 제거 | Perplexity 기준 |
| 중복 제거 | Exact/Near duplicate | MinHash + LSH |
| 유해 콘텐츠 | 혈오 표현, 유해 콘텐츠 | 키워드/분류기 |

In [ ]:
# 간단한 품질 필터링 구현 예시

def quality_filter(text, min_words=10, max_words=100000,
                   min_avg_word_len=2, max_special_ratio=0.3):
    """텍스트 품질 필터링"""
    words = text.split()
    reasons = []

    # 길이 필터
    if len(words) < min_words:
        reasons.append(f"too_short ({len(words)} words)")
    if len(words) > max_words:
        reasons.append(f"too_long ({len(words)} words)")

    # 평균 단어 길이 (스팸/노이즈 감지)
    avg_word_len = np.mean([len(w) for w in words]) if words else 0
    if avg_word_len < min_avg_word_len:
        reasons.append(f"avg_word_too_short ({avg_word_len:.1f})")

    # 특수문자 비율
    special_count = sum(1 for c in text if not c.isalnum() and not c.isspace())
    special_ratio = special_count / len(text) if text else 0
    if special_ratio > max_special_ratio:
        reasons.append(f"too_many_special ({special_ratio:.1%})")

    return len(reasons) == 0, reasons


# 테스트
test_docs = [
    ("오늘 날씨가 좋습니다. 하늘이 맑고 바람이 상쾌합니다. 산책하기 좋은 날입니다. 자연을 만끽하며 힘링을 합니다.", "good"),
    ("안녕", "too_short"),
    ("!!!@@## $$%% &&** (()) [[]]" * 5, "too_many_special"),
    ("a b c d e f g h i j k l m n o", "avg_word_too_short"),
]

print("\ud488\uc9c8 \ud544\ud130\ub9c1 \uacb0\uacfc:")
for text, expected in test_docs:
    passed, reasons = quality_filter(text)
    status = "PASS" if passed else "FAIL"
    reason_str = ', '.join(reasons) if reasons else '-'
    print(f"  [{status}] {text[:40]+'...' if len(text)>40 else text}")
    if reasons:
        print(f"         Reason: {reason_str}")

---
## 3. HuggingFace Datasets 사용법

HuggingFace `datasets` 라이브러리는 LLM 학습에 필수적인 도구다.

### 핵심 기능
- `load_dataset`: Hub에서 데이터셋 다운로드
- `streaming=True`: 대용량 데이터를 스트리밍으로 처리 (메모리 절약)
- `map`: 데이터에 변환 함수 적용
- `filter`: 조건에 맞는 데이터만 선택

In [ ]:
from datasets import load_dataset

# 위키피디아 한국어 데이터셋 로드 (streaming 모드)
# streaming=True로 로드하면 전체 데이터를 다운로드하지 않고 필요한 만큼만 가져옴
print("한국어 Wikipedia 데이터셋 로드 (streaming)...")
wiki_ko = load_dataset("wikipedia", "20220301.ko", streaming=True, split="train",
                        trust_remote_code=True)

# 처음 5개 샘플 확인
print("\n첫 5개 문서:")
samples = []
for i, example in enumerate(wiki_ko):
    if i >= 5:
        break
    samples.append(example)
    title = example['title']
    text_preview = example['text'][:100].replace('\n', ' ')
    text_len = len(example['text'])
    print(f"  [{i}] {title} ({text_len:,} chars): {text_preview}...")

print(f"\n\uceec\ub7fc: {list(samples[0].keys())}")

In [ ]:
# map 예시: 텍스트 길이 계산

def add_text_length(example):
    """text \uae38\uc774\ub97c \ucd94\uac00\ud558\ub294 \ubcc0\ud658 \ud568\uc218"""
    example['text_length'] = len(example['text'])
    example['word_count'] = len(example['text'].split())
    return example


# streaming 데이터셋에 map 적용
wiki_with_length = wiki_ko.map(add_text_length)

# 첫 100개 문서의 길이 분포 확인
lengths = []
word_counts = []
for i, example in enumerate(wiki_with_length):
    if i >= 100:
        break
    lengths.append(example['text_length'])
    word_counts.append(example['word_count'])

print(f"\uccab 100\uac1c \ubb38\uc11c \ud1b5\uacc4:")
print(f"  \ud3c9\uade0 \ubb38\uc790 \uc218: {np.mean(lengths):,.0f}")
print(f"  \uc911\uc559\uac12:      {np.median(lengths):,.0f}")
print(f"  \ucd5c\uc18c/\ucd5c\ub300:   {min(lengths):,} / {max(lengths):,}")
print(f"  \ud3c9\uade0 \ub2e8\uc5b4 \uc218: {np.mean(word_counts):,.0f}")

In [ ]:
# 데이터 길이 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.hist(lengths, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(x=np.median(lengths), color='red', linestyle='--', label=f'Median: {np.median(lengths):,.0f}')
ax.set_xlabel('Text Length (chars)')
ax.set_ylabel('Count')
ax.set_title('Wikipedia (ko): Text Length Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.hist(word_counts, bins=30, color='coral', edgecolor='white', alpha=0.8)
ax.axvline(x=np.median(word_counts), color='red', linestyle='--', label=f'Median: {np.median(word_counts):,.0f}')
ax.set_xlabel('Word Count')
ax.set_ylabel('Count')
ax.set_title('Wikipedia (ko): Word Count Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. 토큰화 파이프라인: 텍스트 -> 토큰 IDs -> 고정 길이 청크

LLM 학습에서는 텍스트를 **고정 길이의 토큰 시퀀스**로 변환해야 한다.

### 파이프라인
```
원델 텍스트 -> 토큰화 -> 토큰 ID 시퀀스 -> 고정 길이 청크 -> DataLoader

"안녕하세요 오늘..."  ->  [128, 9340, ...]  ->  [128, 9340, 23, 456, 789, ..., 0, 0]
                                                     |--- max_length ----|
```

### 청크 전략 2가지

1. **Packing**: 여러 문서를 이어붙여서 max_length로 자르기 (토큰 낭비 없음)
2. **Padding**: 각 문서를 독립적으로 처리, 짧으면 패딩 (토큰 낭비 발생)

LLM 학습에서는 **Packing**이 주로 사용된다.

In [ ]:
from transformers import AutoTokenizer

# GPT-2 토큰나이저 로드
tokenizer = AutoTokenizer.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"\ud1a0\ud070\ub098\uc774\uc800: {tokenizer.__class__.__name__}")
print(f"\uc5b4\ud718 \ud06c\uae30: {tokenizer.vocab_size:,}")
print(f"EOS token: '{tokenizer.eos_token}' (ID: {tokenizer.eos_token_id})")
print(f"PAD token: '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")

In [ ]:
# Packing 방식 구현

def pack_texts_into_chunks(texts, tokenizer, max_length=128):
    """
    \uc5ec\ub7ec \ud14d\uc2a4\ud2b8\ub97c \uc774\uc5b4\ubd99\uc5ec\uc11c \uace0\uc815 \uae38\uc774 \uccad\ud06c\ub85c \ubd84\ud560
    \ud1a0\ud070\uc744 \ub0ad\ube44\ud558\uc9c0 \uc54a\ub294 \ud6a8\uc728\uc801\uc778 \ubc29\ubc95
    """
    # \ubaa8\ub4e0 \ud14d\uc2a4\ud2b8\ub97c \ud1a0\ud070\ud654\ud558\uace0 \uc774\uc5b4\ubd99\uc784
    all_token_ids = []
    for text in texts:
        token_ids = tokenizer.encode(text, add_special_tokens=False)
        all_token_ids.extend(token_ids)
        all_token_ids.append(tokenizer.eos_token_id)  # \ubb38\uc11c \uad6c\ubd84\uc790

    # \uace0\uc815 \uae38\uc774\ub85c \uc790\ub974\uae30
    chunks = []
    for i in range(0, len(all_token_ids) - max_length, max_length):
        chunk = all_token_ids[i:i + max_length]
        chunks.append(chunk)

    return chunks, all_token_ids


# \uc608\uc2dc \ud14d\uc2a4\ud2b8
texts = [
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning is a subset of artificial intelligence.",
    "Large language models have transformed natural language processing.",
    "Training data quality significantly impacts model performance.",
    "Tokenization converts text into numerical representations for the model.",
]

chunks, all_ids = pack_texts_into_chunks(texts, tokenizer, max_length=32)

print(f"\uc6d0\ubcf8 \ud14d\uc2a4\ud2b8 \uc218: {len(texts)}")
print(f"\uc804\uccb4 \ud1a0\ud070 \uc218: {len(all_ids)}")
print(f"\uccad\ud06c \uc218 (max_length=32): {len(chunks)}")
print(f"\ub0ad\ube44\ub41c \ud1a0\ud070: {len(all_ids) - len(chunks) * 32} (\ub9c8\uc9c0\ub9c9 \ubd88\uc644\uc804 \uccad\ud06c)")

print(f"\n\uac01 \uccad\ud06c:")
for i, chunk in enumerate(chunks):
    decoded = tokenizer.decode(chunk)
    print(f"  Chunk {i}: [{len(chunk)} tokens] {decoded[:80]}...")

In [ ]:
# Packing vs Padding 비교 시각화

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Packing
ax = axes[0]
colors = plt.cm.Set3(np.linspace(0, 1, len(texts)))

# 각 텍스트의 토큰 수
token_lengths = [len(tokenizer.encode(t, add_special_tokens=False)) + 1 for t in texts]  # +1 for EOS
max_len = 32

# Packing: 이어붙여서 자르기
x_pos = 0
for i, (tlen, color) in enumerate(zip(token_lengths, colors)):
    chunk_idx = x_pos // max_len
    y = -chunk_idx
    x_in_chunk = x_pos % max_len

    # 청크 경계를 넘는 경우
    remaining_in_chunk = max_len - x_in_chunk
    if tlen <= remaining_in_chunk:
        ax.barh(y, tlen, left=x_in_chunk, color=color, edgecolor='black', height=0.6, alpha=0.8)
        ax.text(x_in_chunk + tlen/2, y, f'T{i+1}', ha='center', va='center', fontsize=8)
    else:
        # 첫 부분
        ax.barh(y, remaining_in_chunk, left=x_in_chunk, color=color, edgecolor='black', height=0.6, alpha=0.8)
        ax.text(x_in_chunk + remaining_in_chunk/2, y, f'T{i+1}', ha='center', va='center', fontsize=8)
        # 나머지
        remaining_tokens = tlen - remaining_in_chunk
        ax.barh(y-1, remaining_tokens, left=0, color=color, edgecolor='black', height=0.6, alpha=0.8)
        ax.text(remaining_tokens/2, y-1, f'T{i+1}', ha='center', va='center', fontsize=8)
    x_pos += tlen

ax.set_xlim(0, max_len)
ax.set_title(f'Packing (max_length={max_len}): No Wasted Tokens')
ax.set_xlabel('Token Position')
ax.set_yticks([])
ax.grid(True, alpha=0.3, axis='x')

# Padding
ax = axes[1]
for i, (tlen, color) in enumerate(zip(token_lengths, colors)):
    # 데이터
    ax.barh(-i, min(tlen, max_len), left=0, color=color, edgecolor='black', height=0.6, alpha=0.8)
    ax.text(min(tlen, max_len)/2, -i, f'T{i+1} ({tlen})', ha='center', va='center', fontsize=8)
    # 패딩
    if tlen < max_len:
        ax.barh(-i, max_len - tlen, left=tlen, color='lightgray', edgecolor='black', height=0.6, alpha=0.5)
        ax.text(tlen + (max_len - tlen)/2, -i, 'PAD', ha='center', va='center', fontsize=8, color='gray')

ax.set_xlim(0, max_len)
ax.set_title(f'Padding (max_length={max_len}): Wasted Tokens (gray)')
ax.set_xlabel('Token Position')
ax.set_yticks([])
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

total_tokens = sum(token_lengths)
padding_waste = sum(max(0, max_len - tlen) for tlen in token_lengths)
print(f"Padding \ubc29\uc2dd\uc758 \ud1a0\ud070 \ub0ad\ube44: {padding_waste}/{len(texts)*max_len} = {padding_waste/(len(texts)*max_len):.1%}")
print(f"Packing \ubc29\uc2dd\uc758 \ud1a0\ud070 \ub0ad\ube44: ~0%")
print(f"-> LLM \ud559\uc2b5\uc5d0\uc11c\ub294 Packing\uc774 \ud6e8\uc52c \ud6a8\uc728\uc801")

---
## 5. DataLoader 구성: 셔플, 배치, 패딩 전략

### LLM 학습용 DataLoader의 핵심 요소

| 요소 | 설명 |
|------|------|
| Shuffle | 데이터 순서를 성어서 학습 안정성 확보 |
| Batching | GPU 활용을 위해 여러 샘플을 모음 |
| Padding/Packing | 같은 길이로 맞춤 |
| Prefetching | 다음 배치를 미리 준비 (I/O 대기 최소화) |

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class LLMPretrainingDataset(Dataset):
    """
    LLM \uc0ac\uc804\ud559\uc2b5\uc6a9 \ub370\uc774\ud130\uc14b
    \ud14d\uc2a4\ud2b8\ub97c \ud1a0\ud070\ud654\ud558\uace0 \uace0\uc815 \uae38\uc774\ub85c \ud328\ud0b9
    """
    def __init__(self, texts, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.chunks = self._prepare_chunks(texts)

    def _prepare_chunks(self, texts):
        """\ud14d\uc2a4\ud2b8\ub97c \ud1a0\ud070\ud654\ud558\uace0 \uace0\uc815 \uae38\uc774 \uccad\ud06c\ub85c \ubd84\ud560"""
        all_ids = []
        for text in texts:
            ids = self.tokenizer.encode(text, add_special_tokens=False)
            all_ids.extend(ids)
            all_ids.append(self.tokenizer.eos_token_id)

        # \uace0\uc815 \uae38\uc774\ub85c \uc790\ub974\uae30
        chunks = []
        for i in range(0, len(all_ids) - self.max_length, self.max_length):
            input_ids = all_ids[i:i + self.max_length]
            # LLM: input = tokens[:-1], target = tokens[1:]
            chunks.append(torch.tensor(input_ids, dtype=torch.long))
        return chunks

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        input_ids = self.chunks[idx]
        # \uc790\uae30\ud68c\uadc0 \ud559\uc2b5: input\uc740 [:-1], target\uc740 [1:]
        return {
            'input_ids': input_ids[:-1],
            'labels': input_ids[1:],
        }


# \ub370\uc774\ud130\uc14b \uc0dd\uc131
sample_texts = [
    "The Transformer architecture has revolutionized natural language processing.",
    "Attention mechanism allows the model to focus on relevant parts of the input.",
    "Large language models are trained on vast amounts of text data from the internet.",
    "Pre-training followed by fine-tuning is the standard approach for modern NLP.",
    "Self-supervised learning enables models to learn from unlabeled data effectively.",
] * 20  # \ubc18\ubcf5\ud574\uc11c \ub370\uc774\ud130 \ub298\ub9ac\uae30

dataset = LLMPretrainingDataset(sample_texts, tokenizer, max_length=64)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

print(f"\ub370\uc774\ud130\uc14b \ud06c\uae30: {len(dataset)} \uccad\ud06c")
print(f"\ubc30\uce58 \uc218: {len(dataloader)}")
print(f"\uccad\ud06c \uae38\uc774: {dataset.max_length}")

# \uccab \ubc88\uc9f8 \ubc30\uce58 \ud655\uc778
batch = next(iter(dataloader))
print(f"\n\uccab \ubc88\uc9f8 \ubc30\uce58:")
print(f"  input_ids shape: {batch['input_ids'].shape}")
print(f"  labels shape:    {batch['labels'].shape}")

# \uccab \uc0d8\ud50c \ub514\ucf54\ub529
print(f"\n  input:  {tokenizer.decode(batch['input_ids'][0][:20])}...")
print(f"  target: {tokenizer.decode(batch['labels'][0][:20])}...")
print(f"  -> target\uc740 input\uc744 \ud55c \ud1a0\ud070\uc529 \uc624\ub978\ucabd\uc73c\ub85c \uc2dc\ud504\ud2b8\ud55c \uac83")

In [ ]:
# 학습 루프 예시 (\uc2e4제 학습 시연)
import torch.nn as nn

# 간단한 Language Model (실제로는 Transformer 사용)
class TinyLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids):
        x = self.embedding(input_ids)  # (B, T) -> (B, T, E)
        x, _ = self.rnn(x)             # (B, T, E) -> (B, T, H)
        logits = self.head(x)          # (B, T, H) -> (B, T, V)
        return logits


model = TinyLM(tokenizer.vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 5 epoch 학습
losses = []
for epoch in range(5):
    epoch_loss = 0
    for batch in dataloader:
        optimizer.zero_grad()
        logits = model(batch['input_ids'])  # (B, T, V)
        # reshape for CrossEntropyLoss
        loss = criterion(logits.view(-1, tokenizer.vocab_size), batch['labels'].view(-1))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}: loss = {avg_loss:.4f}")

# Loss curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(losses)+1), losses, 'b-o', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Tiny LM Training Loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. 소규모 한국어 Corpus 구축

위키피디아 한국어 데이터를 로드하고, 전처리하여 학습용 corpus로 만들어보자.

In [ ]:
import re

def clean_wiki_text(text):
    """위키피디아 \ud14d\uc2a4\ud2b8 \uc804\ucc98\ub9ac"""
    # \uc704\ud0a4 \ub9c8\ud06c\uc5c5 \uc81c\uac70
    text = re.sub(r'\{\{[^}]+\}\}', '', text)
    text = re.sub(r'\[\[[^]]*\|([^]]+)\]\]', r'\1', text)  # [[link|text]] -> text
    text = re.sub(r'\[\[([^]]+)\]\]', r'\1', text)  # [[text]] -> text
    text = re.sub(r'<[^>]+>', '', text)  # HTML tags

    # \ud2b9\uc218\ubb38\uc790 \uc815\ub9ac
    text = re.sub(r'\n{3,}', '\n\n', text)  # \uc5ec\ub7ec \uc904\ubc14\uafc8 -> 2\uac1c
    text = re.sub(r'\s+', ' ', text).strip()  # \uc5f0\uc18d \uacf5\ubc31 \uc81c\uac70

    return text


# \ud55c\uad6d\uc5b4 \uc704\ud0a4\ud53c\ub514\uc544\uc5d0\uc11c 200\uac1c \ubb38\uc11c \ub85c\ub4dc \ubc0f \uc804\ucc98\ub9ac
print("\ud55c\uad6d\uc5b4 Wikipedia \ub370\uc774\ud130 \ub85c\ub4dc \ubc0f \uc804\ucc98\ub9ac...")

wiki_ko_stream = load_dataset("wikipedia", "20220301.ko", streaming=True, split="train",
                               trust_remote_code=True)

cleaned_texts = []
for i, example in enumerate(wiki_ko_stream):
    if i >= 200:
        break

    text = clean_wiki_text(example['text'])

    # \ud488\uc9c8 \ud544\ud130
    if len(text.split()) < 20:  # \ub108\ubb34 \uc9e7\uc740 \ubb38\uc11c \uc81c\uc678
        continue

    cleaned_texts.append(text)

print(f"\uc804\ucc98\ub9ac \uc644\ub8cc: {len(cleaned_texts)}\uac1c \ubb38\uc11c")
print(f"\uc804\uccb4 \ubb38\uc790 \uc218: {sum(len(t) for t in cleaned_texts):,}")
print(f"\uc804\uccb4 \ub2e8\uc5b4 \uc218: {sum(len(t.split()) for t in cleaned_texts):,}")

# \uc0d8\ud50c \ud655\uc778
print(f"\n\uc0d8\ud50c (\uccab 200\uc790):")
print(f"  {cleaned_texts[0][:200]}...")

In [ ]:
# 한국어 corpus 토큰화 및 DataLoader 구성

# GPT-2 토큰나이저로 한국어 토큰화
ko_dataset = LLMPretrainingDataset(cleaned_texts, tokenizer, max_length=128)
ko_dataloader = DataLoader(ko_dataset, batch_size=8, shuffle=True)

print(f"\ud55c\uad6d\uc5b4 Corpus \ud1b5\uacc4:")
print(f"  \ubb38\uc11c \uc218: {len(cleaned_texts)}")
print(f"  \uccad\ud06c \uc218: {len(ko_dataset)}")
print(f"  \ubc30\uce58 \uc218: {len(ko_dataloader)} (batch_size=8)")
print(f"  \uc2dc\ud000\uc2a4 \uae38\uc774: 128")

# \uccab \ubc30\uce58 \ud655\uc778
batch = next(iter(ko_dataloader))
print(f"\n\uccab \ubc30\uce58 shape:")
print(f"  input_ids: {batch['input_ids'].shape}")
print(f"  labels:    {batch['labels'].shape}")

# \uccab \uc0d8\ud50c \ub514\ucf54\ub529
print(f"\n\uccab \uc0d8\ud50c (\uccab 50 tokens):")
print(f"  {tokenizer.decode(batch['input_ids'][0][:50])}")

In [ ]:
# 한국어 corpus 토큰 분포 분석

# 전체 토큰 수집
all_token_ids = []
for text in cleaned_texts[:50]:  # 첫 50개 문서
    ids = tokenizer.encode(text, add_special_tokens=False)
    all_token_ids.extend(ids)

# 빈도 상위 토큰
token_counts = Counter(all_token_ids)
top_tokens = token_counts.most_common(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 상위 20개 토큰
ax = axes[0]
labels = [tokenizer.decode([t_id]) for t_id, _ in top_tokens]
counts = [c for _, c in top_tokens]

bars = ax.barh(range(len(labels)-1, -1, -1), counts, color='steelblue', edgecolor='white')
ax.set_yticks(range(len(labels)-1, -1, -1))
ax.set_yticklabels([repr(l) for l in labels], fontsize=9)
ax.set_xlabel('Frequency')
ax.set_title('Top 20 Tokens (Korean Wikipedia)')
ax.grid(True, alpha=0.3, axis='x')

# 오른쪽: 토큰 길이 분포 (문서당)
ax = axes[1]
doc_token_lengths = [len(tokenizer.encode(t, add_special_tokens=False)) for t in cleaned_texts[:50]]
ax.hist(doc_token_lengths, bins=20, color='coral', edgecolor='white', alpha=0.8)
ax.axvline(x=np.median(doc_token_lengths), color='red', linestyle='--',
           label=f'Median: {np.median(doc_token_lengths):.0f}')
ax.set_xlabel('Token Count per Document')
ax.set_ylabel('Count')
ax.set_title('Token Count Distribution (GPT-2 tokenizer)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\ud1a0\ud070 \ud1b5\uacc4 (\uccab 50\uac1c \ubb38\uc11c):")
print(f"  \uc804\uccb4 \ud1a0\ud070 \uc218: {len(all_token_ids):,}")
print(f"  \uace0\uc720 \ud1a0\ud070 \uc218: {len(set(all_token_ids)):,}")
print(f"  \ubb38\uc11c\ub2f9 \ud3c9\uade0 \ud1a0\ud070: {np.mean(doc_token_lengths):,.0f}")
print(f"  -> GPT-2 \ud1a0\ud070\ub098\uc774\uc800\ub294 \ud55c\uad6d\uc5b4\uc5d0 \ucd5c\uc801\ud654\ub418\uc9c0 \uc54a\uc544 \ud1a0\ud070 \uc218\uac00 \ub9ce\uc74c")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: 완성된 데이터 파이프라인 구축

아래 조건으로 전체 파이프라인을 구축하세요:

1. 한국어 위키피디아에서 500개 문서 로드 (streaming)
2. 텍스트 전처리: `clean_wiki_text` 적용 + 50단어 이상만 선택
3. GPT-2 토큰나이저로 토큰화 (max_length=256)
4. DataLoader 구성 (batch_size=4, shuffle=True)
5. TinyLM 모델로 3 epoch 학습하고 loss curve 시각화

In [ ]:
# TODO: \uc644\uc131\ub41c \ub370\uc774\ud130 \ud30c\uc774\ud504\ub77c\uc778 \uad6c\ucd95
# 1. \ud55c\uad6d\uc5b4 \uc704\ud0a4\ud53c\ub514\uc544 500\uac1c \ubb38\uc11c \ub85c\ub4dc
# 2. clean_wiki_text + \uae38\uc774 \ud544\ud130 (50\ub2e8\uc5b4 \uc774\uc0c1)
# 3. LLMPretrainingDataset(texts, tokenizer, max_length=256)
# 4. DataLoader(dataset, batch_size=4, shuffle=True)
# 5. TinyLM \ubaa8\ub378 3 epoch \ud559\uc2b5 + loss curve


---
## 핵심 정리

| 개념 | 설명 |
|------|------|
| 데이터 소스 | Common Crawl, Wikipedia, Books, Code 등 혼합 |
| 중복 제거 | MinHash + LSH로 유사 문서 발견 및 제거 |
| 품질 필터링 | 길이, 언어, 특수문자 비율 등으로 필터 |
| HuggingFace Datasets | load_dataset, streaming, map, filter |
| Packing | 문서를 이어붙여 고정 길이 청크로 분할 (토큰 낭비 없음) |
| 자기회귀 학습 | input = tokens[:-1], target = tokens[1:] |
| 샘플링 비율 | 고품질 데이터는 여러 번 반복 (Wikipedia: 2.45x) |

**다음 단계**: [project-small-lm](project-small-lm/) - 소규모 LM 직접 학습 프로젝트